# FREE MOBILE GAME LAUNCH STRATEGY ANALYSIS

## Business Context

Stakeholder là một mobile game studio đang chuẩn bị ra mắt một **game Free-to-play** trên App Store.

Studio đã quyết định đi theo mô hình **Free**, nên bài phân tích này không trả lời câu hỏi:

> Nên làm Free hay Paid?

Thay vào đó, bài phân tích tập trung trả lời:

1. Nếu ra mắt một game Free, nên ưu tiên genre nào?
2. Genre nào có tiềm năng tạo traction tốt trong giai đoạn gần nhất?
3. Game Free thành công thường có đặc điểm gì về rating, popularity, tốc độ traction và localization?
4. Nên benchmark game/developer nào?
5. Localization nên triển khai rộng ngay từ đầu hay theo từng giai đoạn?
6. Launch strategy nên nghiêng về viral/casual ngắn hạn hay live-service dài hạn?

## Final Business Decision

Kết quả phân tích sẽ hỗ trợ stakeholder ra quyết định về:

- Genre ưu tiên để phát triển game Free.
- Nhóm game/developer nên benchmark.
- Chiến lược localization ban đầu.
- Launch strategy phù hợp với mục tiêu scale user.


# 1. Business Question Clarification

## Stakeholder muốn biết điều gì?

Stakeholder muốn biết nếu studio ra mắt một game Free thì nên chọn hướng nào để tối đa hóa khả năng tạo traction ban đầu.

## Quyết định kinh doanh cần đưa ra

Sau phân tích, stakeholder cần quyết định:

1. Nên ưu tiên genre nào cho game Free?
2. Nên học hỏi benchmark từ game/developer nào?
3. Có nên localization rộng ngay từ đầu không?
4. Nên đi theo hướng viral/casual ngắn hạn hay live-service dài hạn?

## SMART Objective

Trong phạm vi dataset App Store Games giai đoạn 2008–2019, xác định các genre, game và developer Free có tín hiệu traction tốt nhất trong 3 năm gần nhất của dataset, dựa trên rating count, weighted rating, rating velocity và localization proxy, nhằm hỗ trợ định hướng launch game Free.

## North Star Metric

Vì game Free không thu tiền tải app ban đầu, North Star Metric phù hợp là:

> Traction Quality

Tức là khả năng game vừa tạo được độ phổ biến, vừa có chất lượng rating đáng tin cậy.

## Proxy Metrics

Dataset không có download, revenue, retention, IAP, DAU/MAU nên cần dùng proxy:

| Proxy Metric | Ý nghĩa business | Giới hạn |
|---|---|---|
| rating_count | Proxy cho popularity/traction | Không phải download thật |
| weighted_rating | Rating đáng tin hơn average rating | Vẫn dựa trên review, không phải retention |
| rating_velocity | Tốc độ tạo rating theo thời gian | Không có lịch sử rating theo tháng/năm |
| language_count | Proxy cho localization/global reach | Không biết doanh thu theo quốc gia |
| genre_freshness_rate | Genre còn active gần đây hay không | Không đo trực tiếp market demand |


# 2. Scope & Limitation

## Đơn vị phân tích

Mỗi dòng trong dataset đại diện cho một app/game trên App Store.

## Phạm vi thời gian

Dataset có dữ liệu release date từ 2008 đến 2019.

Tuy nhiên, khi đưa recommendation cho game mới, bài phân tích sẽ ưu tiên **3 năm gần nhất trong dataset**, thường là 2017–2019.

## Scope chính

Chỉ phân tích game Free:

`Price per App (USD) = 0`

Paid game chỉ được dùng để kiểm tra tổng quan ban đầu, không dùng làm nhánh recommendation chính.

## Dataset có thể trả lời

Dataset có thể hỗ trợ trả lời:

- Game Free nào có rating count cao?
- Game Free nào có weighted rating tốt?
- Game Free nào có rating velocity cao?
- Genre nào còn active gần đây?
- Genre nào có traction tốt trong nhóm Free game?
- Developer nào có nhiều game Free có traction tốt?
- Nhóm game traction cao thường hỗ trợ bao nhiêu ngôn ngữ?

## Dataset không thể trả lời trực tiếp

Dataset không có:

- Download count
- Revenue
- IAP revenue
- Ads revenue
- Retention
- DAU/MAU
- Session duration
- Marketing spend
- Rating count history theo từng tháng/năm

Vì vậy các metric như `rating_velocity`, `free_success_score`, `genre_attractiveness_score` chỉ là proxy hỗ trợ ra quyết định, không phải kết luận tuyệt đối.


# 3. Import Library & Load Data

Cell này import thư viện và load file `App Store Games.xlsx`.

Nếu chạy trên Google Colab, hãy upload file Excel khi được yêu cầu.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

file_path = "/content/App Store Games.xlsx"

if not os.path.exists(file_path):
    from google.colab import files
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]

# Load robustly: ưu tiên sheet "App Store Games", nếu không có thì lấy sheet đầu tiên
excel_file = pd.ExcelFile(file_path)
sheet_name = "App Store Games" if "App Store Games" in excel_file.sheet_names else excel_file.sheet_names[0]
df_raw = pd.read_excel(file_path, sheet_name=sheet_name)

print("Sheet used:", sheet_name)
print("Dataset shape:", df_raw.shape)
display(df_raw.head())


# 4. Select Necessary Columns

Chỉ chọn các cột cần thiết cho business request.

| Cột | Lý do sử dụng |
|---|---|
| App ID | Xác định app unique, kiểm tra duplicate |
| Name | Tên game để benchmark |
| Average User Rating | Chất lượng rating ban đầu |
| User Rating Count | Proxy cho popularity/traction |
| Price per App (USD) | Lọc game Free |
| Developer | Benchmark developer |
| Age Rating | Hiểu positioning/tệp người chơi |
| Languages | Phân tích localization |
| Size in Bytes | Đánh giá friction tải app |
| Primary Genre | Genre chính nếu cần |
| Genres | Tách genre game cụ thể |
| Release Date | Phân tích thời gian, freshness, rating velocity |


In [ ]:
rename_map = {
    "App ID": "app_id",
    "Name": "name",
    "Average User Rating": "avg_rating",
    "User Rating Count": "rating_count",
    "Price per App (USD)": "price_usd",
    "Developer": "developer",
    "Age Rating": "age_rating",
    "Languages": "languages",
    "Size in Bytes": "size_bytes",
    "Primary Genre": "primary_genre",
    "Genres": "genres",
    "Release Date": "release_date"
}

data = df_raw.rename(columns=rename_map).copy()

selected_cols = [
    "app_id",
    "name",
    "avg_rating",
    "rating_count",
    "price_usd",
    "developer",
    "age_rating",
    "languages",
    "size_bytes",
    "primary_genre",
    "genres",
    "release_date"
]

missing_cols = [col for col in selected_cols if col not in data.columns]
if missing_cols:
    raise ValueError(f"Các cột sau không có trong dataset: {missing_cols}")

data = data[selected_cols].copy()

display(data.head())


# 5. Analysis Plan

Trước khi code phân tích, cần lập plan để đảm bảo mỗi bảng/biểu đồ đều phục vụ một business question.


In [ ]:
analysis_plan = pd.DataFrame({
    "business_question": [
        "Thị trường Free game thay đổi như thế nào theo thời gian?",
        "Genre nào phù hợp nhất để launch Free game?",
        "Genre nào còn active trong giai đoạn gần nhất?",
        "Game Free nào tạo traction nhanh nhất?",
        "Game Free nào có rating đáng tin cậy nhất?",
        "Game Free thành công có đặc điểm gì?",
        "Localization nên làm rộng ngay hay theo giai đoạn?",
        "Developer nào nên benchmark?",
        "Game cụ thể nào nên lấy làm case study?",
        "Stakeholder nên hành động gì tiếp theo?"
    ],
    "metric_formula": [
        "free_app_count, rated_app_count, total_rating_count theo release_year",
        "free_genre_attractiveness_score = weighted score từ rating_velocity, weighted_rating, rating_count, freshness",
        "genre_freshness_rate = recent_free_apps / total_free_apps",
        "rating_velocity = rating_count / game_age_years",
        "Bayesian weighted_rating",
        "free_success_score = weighted score từ rating_count, weighted_rating, rating_velocity, language_count",
        "language_count, top languages among high-traction games",
        "developer_benchmark_score",
        "Top game theo free_success_score",
        "Top 2–3 actions dựa trên output"
    ],
    "needed_columns": [
        "price_usd, release_date, rating_count",
        "genres, release_date, rating_count, avg_rating, languages",
        "genres, release_date, app_id",
        "name, developer, release_date, rating_count",
        "avg_rating, rating_count",
        "name, developer, rating_count, weighted_rating, rating_velocity, languages",
        "languages, rating_velocity",
        "developer, rating_count, weighted_rating, rating_velocity, languages",
        "name, developer, genre, rating_count, rating_velocity, languages",
        "All final outputs"
    ],
    "expected_output": [
        "1 summary table + 1 line chart",
        "1 genre ranking table + 1 bar chart",
        "1 freshness table",
        "1 top game table + 1 bar chart",
        "Weighted rating used inside ranking",
        "1 benchmark game table",
        "1 localization table + 1 language table",
        "1 developer benchmark table",
        "1 case study table",
        "Markdown recommendation"
    ]
})

display(analysis_plan)


# 6. Data Cleaning & Feature Engineering

Bước này chỉ làm sạch dữ liệu ở mức đủ dùng.

Các xử lý chính:

1. Chuyển numeric columns về dạng số.
2. Chuyển release_date về datetime.
3. Xóa duplicate theo app_id.
4. Tạo release_year.
5. Tạo price_type để lọc Free game.
6. Tạo game_genre từ Genres.
7. Tạo language_count từ Languages.
8. Tạo size_mb từ Size in Bytes.

Không preprocessing quá mức vì mục tiêu là phân tích để ra quyết định, không phải xây model.


In [ ]:
# Convert numeric columns
numeric_cols = ["avg_rating", "rating_count", "price_usd", "size_bytes"]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Convert datetime robustly: xử lý cả kiểu datetime và Excel serial number
if pd.api.types.is_numeric_dtype(data["release_date"]):
    data["release_date"] = pd.to_datetime(
        data["release_date"],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )
else:
    data["release_date"] = pd.to_datetime(data["release_date"], errors="coerce")

# Data quality before dedup
before_dedup = len(data)
duplicate_app_count = data["app_id"].duplicated().sum()

# Remove duplicate app_id
data = data.drop_duplicates(subset=["app_id"]).copy()

# Remove rows missing key fields required for scope
data = data.dropna(subset=["app_id", "price_usd", "release_date"]).copy()

# Create release year
data["release_year"] = data["release_date"].dt.year.astype(int)

# Create price type
data["price_type"] = np.where(data["price_usd"].eq(0), "Free", "Paid")

# Parse list-like text columns
def parse_list(x):
    if pd.isna(x):
        return []
    return [i.strip() for i in str(x).split(",") if i.strip()]

GAME_GENRES = {
    "Action", "Adventure", "Arcade", "Board", "Card", "Casino", "Casual",
    "Family", "Music", "Puzzle", "Racing", "Role Playing", "Simulation",
    "Sports", "Strategy", "Trivia", "Word"
}

def extract_game_genre(genres_text, primary_genre):
    genre_list = parse_list(genres_text)
    
    # Ưu tiên genre game cụ thể trong cột Genres
    for g in genre_list:
        if g in GAME_GENRES:
            return g
    
    # Nếu không thấy, dùng Primary Genre nếu nằm trong game genre
    if pd.notna(primary_genre) and primary_genre in GAME_GENRES:
        return primary_genre
    
    # Nếu vẫn không có, lấy genre không quá generic
    for g in genre_list:
        if g not in ["Games", "Entertainment"]:
            return g
    
    return np.nan

def parse_languages(x):
    if pd.isna(x):
        return []
    return sorted(set([i.strip().upper() for i in str(x).split(",") if i.strip()]))

data["game_genre"] = data.apply(
    lambda row: extract_game_genre(row["genres"], row["primary_genre"]),
    axis=1
)

data["language_list"] = data["languages"].apply(parse_languages)
data["language_count"] = data["language_list"].apply(len)
data["has_english"] = data["language_list"].apply(lambda x: "EN" in x)

data["size_mb"] = data["size_bytes"] / (1024 ** 2)

display(data.head())


# 7. Data Quality Check

Kiểm tra nhanh chất lượng dữ liệu sau khi cleaning:

- Số dòng còn lại
- Duplicate theo app_id
- Missing ở các cột quan trọng
- Khoảng thời gian dataset
- Tỷ trọng Free/Paid


In [ ]:
quality_check = pd.DataFrame({
    "metric": [
        "rows_before_dedup",
        "duplicate_app_id_before_cleaning",
        "rows_after_cleaning",
        "unique_apps",
        "min_release_year",
        "max_release_year",
        "missing_avg_rating",
        "missing_rating_count",
        "missing_game_genre",
        "missing_languages"
    ],
    "value": [
        before_dedup,
        duplicate_app_count,
        len(data),
        data["app_id"].nunique(),
        data["release_year"].min(),
        data["release_year"].max(),
        data["avg_rating"].isna().sum(),
        data["rating_count"].isna().sum(),
        data["game_genre"].isna().sum(),
        data["languages"].isna().sum()
    ]
})

display(quality_check)

price_type_summary = (
    data.groupby("price_type")
    .agg(
        app_count=("app_id", "nunique"),
        avg_rating=("avg_rating", "mean"),
        total_rating_count=("rating_count", "sum"),
        median_rating_count=("rating_count", "median")
    )
    .reset_index()
)

price_type_summary["app_share"] = price_type_summary["app_count"] / price_type_summary["app_count"].sum()

display(price_type_summary)


# 8. Filter Free Games & Recent Free Games

Từ bước này, phân tích chính chỉ dùng game Free.

Ta tạo 2 dataset:

1. `free_data`: toàn bộ game Free.
2. `recent_free`: game Free trong 3 năm gần nhất của dataset.

Lý do dùng giai đoạn gần nhất:

Dataset kéo dài từ 2008 đến 2019. Nếu dùng toàn bộ lịch sử để recommend cho game mới, kết luận có thể bị ảnh hưởng bởi trend cũ.


In [ ]:
free_data = data[data["price_type"].eq("Free")].copy()

latest_year = int(free_data["release_year"].max())
recent_start_year = latest_year - 2

recent_free = free_data[
    free_data["release_year"].between(recent_start_year, latest_year)
].copy()

display(Markdown(f"""
## Free Game Scope

- Latest year in dataset: **{latest_year}**
- Recent analysis window: **{recent_start_year}–{latest_year}**
- Total Free games: **{len(free_data):,}**
- Recent Free games: **{len(recent_free):,}**
"""))

display(recent_free.head())


# 9. Create Core Free Game KPIs

## 9.1 Weighted Rating

Không nên dùng Average User Rating đơn thuần để xếp hạng game.

Ví dụ:

- Game A: 5 sao nhưng chỉ có 10 reviews.
- Game B: 4.5 sao nhưng có 1,000,000 reviews.

Game B đáng tin hơn Game A.

Vì vậy ta dùng Bayesian Weighted Rating:

`weighted_rating = (v / (v + m)) * R + (m / (v + m)) * C`

Trong đó:

- `R`: average rating của game
- `v`: rating count của game
- `C`: average rating trung bình của Free games
- `m`: ngưỡng rating count tối thiểu, lấy theo percentile 75%

## 9.2 Rating Velocity

Dataset không có lịch sử rating count theo từng tháng/năm, nên không thể tính growth thật.

Ta dùng proxy:

`rating_velocity = rating_count / game_age_years`

Ý nghĩa:

> Trung bình mỗi năm game tạo được bao nhiêu rating count.


In [ ]:
rated_free_mask = (
    free_data["rating_count"].fillna(0).gt(0)
    & free_data["avg_rating"].notna()
)

C = free_data.loc[rated_free_mask, "avg_rating"].mean()
m = free_data.loc[rated_free_mask, "rating_count"].quantile(0.75)
m = max(1, m)

def add_free_game_kpis(df, C, m, latest_year):
    df = df.copy()
    
    votes = df["rating_count"].fillna(0)
    rating = df["avg_rating"].fillna(C)
    
    df["weighted_rating"] = (
        (votes / (votes + m)) * rating
        + (m / (votes + m)) * C
    )
    
    df["game_age_years"] = (latest_year - df["release_year"] + 1).clip(lower=1)
    df["rating_velocity"] = votes / df["game_age_years"]
    
    return df

free_data = add_free_game_kpis(free_data, C, m, latest_year)
recent_free = add_free_game_kpis(recent_free, C, m, latest_year)

# Tạo subset chỉ gồm game có rating_count > 0 để phân tích traction
rated_recent_free = recent_free[recent_free["rating_count"].fillna(0).gt(0)].copy()

display(Markdown(f"""
## KPI Parameters

- Market average rating for Free games, C: **{C:.2f}**
- Minimum rating count threshold, m: **{m:,.0f}**
- Recent Free games with rating count > 0: **{len(rated_recent_free):,}**
"""))

display(
    rated_recent_free[
        [
            "name",
            "developer",
            "game_genre",
            "release_year",
            "avg_rating",
            "rating_count",
            "weighted_rating",
            "game_age_years",
            "rating_velocity",
            "language_count"
        ]
    ].head()
)


# 10. Output 1 — Free Game Market Overview

## Business Question

Thị trường Free game thay đổi như thế nào theo thời gian?

## Metric

- `free_app_count`: số game Free phát hành theo năm.
- `rated_free_app_count`: số game Free có rating.
- `total_rating_count`: tổng rating count.
- `median_rating_velocity`: median rating velocity của các game có rating.

## Business Meaning

Nếu số lượng Free game tăng mạnh, thị trường có thể hấp dẫn nhưng cạnh tranh cao hơn.

Nếu rating count/rating velocity cao, điều đó cho thấy người chơi có tương tác và game có traction.


In [ ]:
yearly_market = (
    recent_free.groupby("release_year")
    .agg(
        free_app_count=("app_id", "nunique"),
        total_rating_count=("rating_count", "sum")
    )
    .reset_index()
)

yearly_rated = (
    rated_recent_free.groupby("release_year")
    .agg(
        rated_free_app_count=("app_id", "nunique"),
        median_rating_velocity=("rating_velocity", "median"),
        median_weighted_rating=("weighted_rating", "median")
    )
    .reset_index()
)

yearly_market = yearly_market.merge(yearly_rated, on="release_year", how="left")

display(yearly_market)

plt.figure(figsize=(10, 5))
plt.plot(yearly_market["release_year"], yearly_market["free_app_count"], marker="o")
plt.title("Recent Free Game Releases by Year")
plt.xlabel("Release Year")
plt.ylabel("Free App Count")
plt.grid(True, alpha=0.3)
plt.show()

last_year = yearly_market.sort_values("release_year").iloc[-1]
display(Markdown(f"""
## Nhận xét

Trong giai đoạn **{recent_start_year}–{latest_year}**, số lượng game Free ra mắt được dùng để đánh giá mức độ cạnh tranh gần đây.

Năm gần nhất trong dataset là **{int(last_year["release_year"])}**, có **{int(last_year["free_app_count"]):,}** game Free ra mắt.

Điều này quan trọng vì nếu thị trường Free game có nhiều app mới, studio cần chọn genre không chỉ có demand mà còn phải cân nhắc cạnh tranh và khả năng tạo traction nhanh.
"""))


# 11. Output 2 — Genre Freshness Rate

## Business Question

Genre nào còn active gần đây?

## Metric

`genre_freshness_rate = recent_free_apps / total_free_apps`

Ý nghĩa:

- Freshness cao: genre vẫn còn nhiều game mới ra mắt.
- Freshness thấp: genre có thể đã cũ hoặc ít được studio mới tham gia.
- Freshness cao + app count cao: thị trường active nhưng cạnh tranh mạnh.


In [ ]:
genre_freshness = (
    free_data.groupby("game_genre")
    .agg(
        total_free_apps=("app_id", "nunique"),
        recent_free_apps_all=("release_year", lambda x: x.ge(recent_start_year).sum())
    )
    .reset_index()
)

genre_freshness["genre_freshness_rate"] = (
    genre_freshness["recent_free_apps_all"] / genre_freshness["total_free_apps"]
)

genre_freshness = genre_freshness.sort_values("genre_freshness_rate", ascending=False)

display(genre_freshness.head(15))

top_fresh_genre = genre_freshness.iloc[0]

display(Markdown(f"""
## Nhận xét

Genre có freshness rate cao nhất là **{top_fresh_genre["game_genre"]}** với freshness rate khoảng **{top_fresh_genre["genre_freshness_rate"]:.1%}**.

Điều này cho thấy genre này vẫn có nhiều game Free mới ra mắt trong giai đoạn gần nhất. Tuy nhiên, freshness cao không đồng nghĩa chắc chắn nên chọn, vì cần kết hợp thêm traction, rating quality và competition.
"""))


# 12. Output 3 — Genre Attractiveness for Free Game

## Business Question

Nếu ra mắt game Free, nên ưu tiên genre nào?

## Metric

Tạo `free_genre_attractiveness_score`.

Đây là analyst-defined score, không phải machine learning model.

Score được tính từ:

| Component | Weight | Ý nghĩa |
|---|---:|---|
| median_rating_velocity | 35% | Genre có tạo traction nhanh không |
| avg_weighted_rating | 25% | Chất lượng rating có đáng tin không |
| total_rating_count | 20% | Tổng popularity/traction |
| genre_freshness_rate | 15% | Genre còn active gần đây không |
| median_language_count | 5% | Có xu hướng localization/global reach không |

Lưu ý:

- Traction metrics chỉ tính trên game có rating count > 0.
- Competition/app count vẫn tính trên toàn bộ recent Free games.


In [ ]:
def minmax_norm(s):
    s = s.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
    
    if s.max() == s.min():
        return pd.Series(0.5, index=s.index)
    
    return (s - s.min()) / (s.max() - s.min())

# Recent app count = competition proxy
genre_recent_count = (
    recent_free.groupby("game_genre")
    .agg(
        recent_free_apps=("app_id", "nunique")
    )
    .reset_index()
)

# Rated recent apps = traction/quality analysis base
genre_recent_quality = (
    rated_recent_free.groupby("game_genre")
    .agg(
        rated_recent_free_apps=("app_id", "nunique"),
        avg_weighted_rating=("weighted_rating", "mean"),
        median_rating_count=("rating_count", "median"),
        total_rating_count=("rating_count", "sum"),
        median_rating_velocity=("rating_velocity", "median"),
        median_language_count=("language_count", "median"),
        english_coverage=("has_english", "mean")
    )
    .reset_index()
)

genre_score = (
    genre_recent_count
    .merge(genre_recent_quality, on="game_genre", how="left")
    .merge(genre_freshness, on="game_genre", how="left")
)

# Chỉ lấy genre có đủ số game có rating để tránh sample quá nhỏ
min_rated_apps = 30
genre_score = genre_score[
    genre_score["rated_recent_free_apps"].fillna(0) >= min_rated_apps
].copy()

# Normalize skewed metrics
genre_score["log_total_rating_count"] = np.log1p(genre_score["total_rating_count"].fillna(0))
genre_score["log_median_rating_velocity"] = np.log1p(genre_score["median_rating_velocity"].fillna(0))

genre_score["rating_velocity_norm"] = minmax_norm(genre_score["log_median_rating_velocity"])
genre_score["weighted_rating_norm"] = minmax_norm(genre_score["avg_weighted_rating"])
genre_score["total_rating_count_norm"] = minmax_norm(genre_score["log_total_rating_count"])
genre_score["freshness_rate_norm"] = minmax_norm(genre_score["genre_freshness_rate"])
genre_score["language_count_norm"] = minmax_norm(genre_score["median_language_count"])

genre_score["free_genre_attractiveness_score"] = (
    0.35 * genre_score["rating_velocity_norm"]
    + 0.25 * genre_score["weighted_rating_norm"]
    + 0.20 * genre_score["total_rating_count_norm"]
    + 0.15 * genre_score["freshness_rate_norm"]
    + 0.05 * genre_score["language_count_norm"]
) * 100

genre_score = genre_score.sort_values(
    "free_genre_attractiveness_score",
    ascending=False
)

genre_attractiveness_table = genre_score[
    [
        "game_genre",
        "recent_free_apps",
        "rated_recent_free_apps",
        "avg_weighted_rating",
        "median_rating_count",
        "total_rating_count",
        "median_rating_velocity",
        "genre_freshness_rate",
        "median_language_count",
        "free_genre_attractiveness_score"
    ]
].head(15)

display(genre_attractiveness_table)

plt.figure(figsize=(11, 6))
plot_data = genre_attractiveness_table.sort_values(
    "free_genre_attractiveness_score",
    ascending=True
)

plt.barh(
    plot_data["game_genre"],
    plot_data["free_genre_attractiveness_score"]
)
plt.title("Top Genres for Free Game Launch")
plt.xlabel("Free Genre Attractiveness Score")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

best_genre = genre_score.iloc[0]

display(Markdown(f"""
## Nhận xét

Genre đứng đầu theo Free Genre Attractiveness Score là **{best_genre["game_genre"]}** với score **{best_genre["free_genre_attractiveness_score"]:.1f}**.

Genre này nổi bật vì kết hợp được nhiều yếu tố: rating velocity, weighted rating, total rating count và freshness rate.

Về mặt business, đây là genre nên được ưu tiên nghiên cứu sâu nếu stakeholder muốn chọn hướng có khả năng tạo traction tốt cho game Free.
"""))


# 13. Output 4 — Genre Opportunity Matrix

## Business Question

Genre nào vừa có traction tốt vừa không quá crowded?

## Logic

Không phải genre score cao là chọn ngay.

Stakeholder cần hiểu genre đó thuộc nhóm nào:

| Nhóm | Ý nghĩa |
|---|---|
| Priority opportunity | Traction cao, cạnh tranh chưa quá cao |
| Big market / high competition | Traction cao nhưng cạnh tranh mạnh |
| Crowded / weak traction | Nhiều app nhưng traction thấp |
| Niche / validate carefully | Quy mô nhỏ hơn, cần test kỹ |

## Metric

- Competition proxy: `recent_free_apps`
- Traction proxy: `median_rating_velocity`


In [ ]:
competition_cut = genre_score["recent_free_apps"].quantile(0.75)
traction_cut = genre_score["median_rating_velocity"].quantile(0.75)

def classify_opportunity(row):
    if row["median_rating_velocity"] >= traction_cut and row["recent_free_apps"] < competition_cut:
        return "Priority opportunity"
    elif row["median_rating_velocity"] >= traction_cut and row["recent_free_apps"] >= competition_cut:
        return "Big market / high competition"
    elif row["median_rating_velocity"] < traction_cut and row["recent_free_apps"] >= competition_cut:
        return "Crowded / weak traction"
    else:
        return "Niche / validate carefully"

genre_score["opportunity_type"] = genre_score.apply(classify_opportunity, axis=1)

genre_opportunity_table = genre_score[
    [
        "game_genre",
        "recent_free_apps",
        "median_rating_velocity",
        "avg_weighted_rating",
        "genre_freshness_rate",
        "free_genre_attractiveness_score",
        "opportunity_type"
    ]
].sort_values("free_genre_attractiveness_score", ascending=False)

display(genre_opportunity_table)

plt.figure(figsize=(10, 6))

plt.scatter(
    genre_score["recent_free_apps"],
    genre_score["median_rating_velocity"],
    s=np.sqrt(genre_score["total_rating_count"].clip(lower=1)) * 2,
    alpha=0.6
)

for _, row in genre_score.head(10).iterrows():
    plt.text(
        row["recent_free_apps"],
        row["median_rating_velocity"],
        row["game_genre"],
        fontsize=9
    )

plt.axvline(competition_cut, linestyle="--", alpha=0.5)
plt.axhline(traction_cut, linestyle="--", alpha=0.5)

plt.title("Genre Opportunity Matrix for Free Games")
plt.xlabel("Competition Proxy: Recent Free App Count")
plt.ylabel("Traction Proxy: Median Rating Velocity")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

priority_genres = genre_opportunity_table[
    genre_opportunity_table["opportunity_type"].eq("Priority opportunity")
]

if len(priority_genres) > 0:
    priority_text = ", ".join(priority_genres["game_genre"].head(3).tolist())
else:
    priority_text = "Không có genre nào rõ ràng thuộc nhóm Priority opportunity"

display(Markdown(f"""
## Nhận xét

Nhóm genre ưu tiên theo ma trận cơ hội là: **{priority_text}**.

Nếu nguồn lực launch có giới hạn, stakeholder nên ưu tiên genre có traction cao nhưng cạnh tranh chưa quá crowded.

Nếu chọn genre thuộc nhóm Big Market / High Competition, studio cần chuẩn bị năng lực marketing, product quality và live-ops tốt hơn.
"""))


# 14. Output 5 — Top Free Benchmark Games

## Business Question

Game Free nào nên dùng để benchmark?

## Metric

Tạo `free_success_score`.

Đây là analyst-defined score, gồm:

| Component | Weight | Ý nghĩa |
|---|---:|---|
| rating_count | 40% | Popularity/traction |
| weighted_rating | 25% | Chất lượng đáng tin |
| rating_velocity | 25% | Tốc độ tạo traction |
| language_count | 10% | Localization/global reach |

Chỉ ranking các game có rating count > 0 để tránh game chưa có traction.


In [ ]:
score_base = rated_recent_free.copy()

score_base["log_rating_count"] = np.log1p(score_base["rating_count"].fillna(0))
score_base["log_rating_velocity"] = np.log1p(score_base["rating_velocity"].fillna(0))

score_base["rating_count_norm"] = minmax_norm(score_base["log_rating_count"])
score_base["weighted_rating_norm"] = minmax_norm(score_base["weighted_rating"])
score_base["rating_velocity_norm"] = minmax_norm(score_base["log_rating_velocity"])
score_base["language_count_norm"] = minmax_norm(score_base["language_count"])

score_base["free_success_score"] = (
    0.40 * score_base["rating_count_norm"]
    + 0.25 * score_base["weighted_rating_norm"]
    + 0.25 * score_base["rating_velocity_norm"]
    + 0.10 * score_base["language_count_norm"]
) * 100

top_free_games = (
    score_base.sort_values("free_success_score", ascending=False)
    [
        [
            "name",
            "developer",
            "game_genre",
            "release_year",
            "avg_rating",
            "rating_count",
            "weighted_rating",
            "rating_velocity",
            "language_count",
            "free_success_score"
        ]
    ]
    .head(20)
)

display(top_free_games)

plt.figure(figsize=(12, 6))
plot_data = top_free_games.head(10).sort_values("free_success_score", ascending=True)

plt.barh(plot_data["name"], plot_data["free_success_score"])
plt.title("Top Free Benchmark Games by Success Score")
plt.xlabel("Free Success Score")
plt.ylabel("Game")
plt.tight_layout()
plt.show()

best_game = top_free_games.iloc[0]

display(Markdown(f"""
## Nhận xét

Game benchmark tốt nhất theo Free Success Score là **{best_game["name"]}** của **{best_game["developer"]}**.

Game này đáng benchmark vì không chỉ có rating count cao, mà còn có weighted rating và rating velocity tốt.

Về mặt business, stakeholder có thể học từ game này về genre positioning, scale strategy và localization footprint.
"""))


# 15. Output 6 — Localization Strategy

## Business Question

Localization nên triển khai rộng ngay từ đầu hay theo từng giai đoạn?

## Logic

Với Free game, localization có thể giúp scale user, nhưng không nên đầu tư quá rộng ngay từ đầu nếu chưa có traction.

Ta phân tích:

- Nhóm game có bao nhiêu ngôn ngữ.
- Nhóm nào có median rating velocity tốt hơn.
- Các ngôn ngữ phổ biến trong nhóm high-traction games.

## Business Interpretation

- Nếu game casual/viral: nên bắt đầu với ít ngôn ngữ chính.
- Nếu game live-service/RPG/strategy/online: có thể cần localization rộng hơn sau khi validate traction.


In [ ]:
def language_group(n):
    if n <= 1:
        return "1 language"
    elif n <= 5:
        return "2-5 languages"
    elif n <= 10:
        return "6-10 languages"
    else:
        return "11+ languages"

rated_recent_free["language_group"] = rated_recent_free["language_count"].apply(language_group)

localization_summary = (
    rated_recent_free.groupby("language_group", observed=False)
    .agg(
        rated_app_count=("app_id", "nunique"),
        avg_weighted_rating=("weighted_rating", "mean"),
        median_rating_count=("rating_count", "median"),
        median_rating_velocity=("rating_velocity", "median"),
        total_rating_count=("rating_count", "sum")
    )
    .reset_index()
)

language_order = ["1 language", "2-5 languages", "6-10 languages", "11+ languages"]

localization_summary["language_group"] = pd.Categorical(
    localization_summary["language_group"],
    categories=language_order,
    ordered=True
)

localization_summary = localization_summary.sort_values("language_group")

display(localization_summary)

plt.figure(figsize=(10, 5))
plt.bar(
    localization_summary["language_group"].astype(str),
    localization_summary["median_rating_velocity"]
)
plt.title("Median Rating Velocity by Localization Group")
plt.xlabel("Localization Group")
plt.ylabel("Median Rating Velocity")
plt.tight_layout()
plt.show()

best_lang_group = localization_summary.sort_values(
    "median_rating_velocity",
    ascending=False
).iloc[0]

display(Markdown(f"""
## Nhận xét

Nhóm localization có median rating velocity cao nhất là **{best_lang_group["language_group"]}**.

Điều này không có nghĩa cứ thêm nhiều ngôn ngữ là game sẽ thành công. Đây chỉ là pattern quan sát được trong nhóm game Free có rating.

Về mặt business, recommendation hợp lý là localization theo từng giai đoạn:

- Phase 1: chọn một số ngôn ngữ chính.
- Phase 2: mở rộng thêm khi game có traction thật.
"""))


# 16. Output 7 — Top Languages Among High-Traction Free Games

## Business Question

Nếu launch Free game, nên ưu tiên ngôn ngữ nào ở phase đầu?

## Logic

Lấy nhóm game có `rating_velocity` nằm trong top 10% để xem các game traction cao thường hỗ trợ ngôn ngữ nào.

Lưu ý:

Đây là pattern mô tả, không phải quan hệ nhân quả.


In [ ]:
high_traction_cut = rated_recent_free["rating_velocity"].quantile(0.90)

high_traction_games = rated_recent_free[
    rated_recent_free["rating_velocity"] >= high_traction_cut
].copy()

language_counter = {}

for langs in high_traction_games["language_list"]:
    for lang in langs:
        language_counter[lang] = language_counter.get(lang, 0) + 1

top_languages_high_traction = (
    pd.DataFrame(
        language_counter.items(),
        columns=["language", "high_traction_game_count"]
    )
    .sort_values("high_traction_game_count", ascending=False)
    .head(15)
)

display(top_languages_high_traction)

plt.figure(figsize=(10, 5))
plot_data = top_languages_high_traction.sort_values(
    "high_traction_game_count",
    ascending=True
)

plt.barh(plot_data["language"], plot_data["high_traction_game_count"])
plt.title("Top Languages Among High-Traction Free Games")
plt.xlabel("Number of High-Traction Games")
plt.ylabel("Language")
plt.tight_layout()
plt.show()

top_language_list = ", ".join(top_languages_high_traction["language"].head(5).tolist())

display(Markdown(f"""
## Nhận xét

Các ngôn ngữ xuất hiện nhiều nhất trong nhóm high-traction Free games là: **{top_language_list}**.

Về mặt business, đây là nhóm ngôn ngữ nên cân nhắc cho localization phase 1.

Tuy nhiên, lựa chọn cuối cùng còn phụ thuộc vào thị trường mục tiêu, ngân sách localization và thể loại game.
"""))


# 17. Output 8 — Developer Benchmark

## Business Question

Developer nào nên benchmark nếu studio muốn launch game Free?

## Metric

Tạo `developer_benchmark_score`.

Score gồm:

| Component | Weight | Ý nghĩa |
|---|---:|---|
| total_rating_count | 35% | Tổng traction |
| median_rating_velocity | 30% | Tốc độ tạo traction |
| avg_weighted_rating | 25% | Chất lượng rating đáng tin |
| median_language_count | 10% | Localization/global reach |

Chỉ lấy developer có ít nhất 2 game Free có rating trong giai đoạn gần nhất để tránh benchmark từ sample quá nhỏ.


In [ ]:
developer_summary = (
    rated_recent_free.groupby("developer")
    .agg(
        rated_free_app_count=("app_id", "nunique"),
        avg_weighted_rating=("weighted_rating", "mean"),
        total_rating_count=("rating_count", "sum"),
        median_rating_count=("rating_count", "median"),
        median_rating_velocity=("rating_velocity", "median"),
        median_language_count=("language_count", "median"),
        main_genre=("game_genre", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan)
    )
    .reset_index()
)

developer_summary = developer_summary[
    developer_summary["rated_free_app_count"] >= 2
].copy()

developer_summary["log_total_rating_count"] = np.log1p(
    developer_summary["total_rating_count"].fillna(0)
)

developer_summary["log_median_rating_velocity"] = np.log1p(
    developer_summary["median_rating_velocity"].fillna(0)
)

developer_summary["total_rating_count_norm"] = minmax_norm(
    developer_summary["log_total_rating_count"]
)

developer_summary["rating_velocity_norm"] = minmax_norm(
    developer_summary["log_median_rating_velocity"]
)

developer_summary["weighted_rating_norm"] = minmax_norm(
    developer_summary["avg_weighted_rating"]
)

developer_summary["language_count_norm"] = minmax_norm(
    developer_summary["median_language_count"]
)

developer_summary["developer_benchmark_score"] = (
    0.35 * developer_summary["total_rating_count_norm"]
    + 0.30 * developer_summary["rating_velocity_norm"]
    + 0.25 * developer_summary["weighted_rating_norm"]
    + 0.10 * developer_summary["language_count_norm"]
) * 100

top_developers = (
    developer_summary.sort_values("developer_benchmark_score", ascending=False)
    [
        [
            "developer",
            "main_genre",
            "rated_free_app_count",
            "avg_weighted_rating",
            "total_rating_count",
            "median_rating_velocity",
            "median_language_count",
            "developer_benchmark_score"
        ]
    ]
    .head(20)
)

display(top_developers)

best_developer = top_developers.iloc[0]

display(Markdown(f"""
## Nhận xét

Developer đứng đầu benchmark score là **{best_developer["developer"]}**, với main genre là **{best_developer["main_genre"]}**.

Developer này đáng benchmark vì có nhiều tín hiệu tốt về traction, rating quality và rating velocity.

Về mặt business, stakeholder nên phân tích sâu portfolio của developer này để học cách chọn genre, đặt positioning và triển khai launch strategy.
"""))


# 18. Output 9 — Case Study: Best Benchmark Free Game

## Business Question

Một game Free tiêu biểu có đặc điểm gì?

## Logic

Chọn game có `free_success_score` cao nhất trong giai đoạn gần nhất để làm case study.

Case study giúp stakeholder hiểu rõ:

- Game thuộc genre nào?
- Rating có đáng tin không?
- Rating count cao không?
- Rating velocity cao không?
- Hỗ trợ bao nhiêu ngôn ngữ?
- Có phù hợp với chiến lược viral hay live-service không?


In [ ]:
case_study_game = (
    score_base.sort_values("free_success_score", ascending=False)
    .head(1)
    .copy()
)

case_study_cols = [
    "name",
    "developer",
    "game_genre",
    "release_year",
    "age_rating",
    "avg_rating",
    "rating_count",
    "weighted_rating",
    "game_age_years",
    "rating_velocity",
    "language_count",
    "languages",
    "size_mb",
    "free_success_score"
]

display(case_study_game[case_study_cols])

case = case_study_game.iloc[0]

display(Markdown(f"""
## Case Study Summary

Game tiêu biểu được chọn là **{case["name"]}** của **{case["developer"]}**.

- Genre: **{case["game_genre"]}**
- Release year: **{int(case["release_year"])}**
- Average rating: **{case["avg_rating"]:.2f}**
- Weighted rating: **{case["weighted_rating"]:.2f}**
- Rating count: **{case["rating_count"]:,.0f}**
- Rating velocity: **{case["rating_velocity"]:,.0f} ratings/year**
- Language count: **{int(case["language_count"])}**
- Free success score: **{case["free_success_score"]:.1f}**

Về mặt business, đây là game nên được dùng để benchmark vì nó có tín hiệu tốt về popularity, chất lượng rating và tốc độ traction.
"""))


# 19. Final Insight & Recommendation

Phần này tổng hợp các output chính để stakeholder ra quyết định.

Recommendation phải gắn với hành động kinh doanh, không chỉ mô tả số liệu.


In [ ]:
top_3_genres = genre_score.head(3)[
    [
        "game_genre",
        "recent_free_apps",
        "rated_recent_free_apps",
        "median_rating_velocity",
        "avg_weighted_rating",
        "genre_freshness_rate",
        "free_genre_attractiveness_score",
        "opportunity_type"
    ]
]

top_5_games = top_free_games.head(5)
top_5_developers = top_developers.head(5)
top_5_languages = top_languages_high_traction.head(5)

display(Markdown("## Top 3 Genre Candidates"))
display(top_3_genres)

display(Markdown("## Top 5 Benchmark Free Games"))
display(top_5_games)

display(Markdown("## Top 5 Benchmark Developers"))
display(top_5_developers)

display(Markdown("## Top 5 Languages for Localization Phase 1"))
display(top_5_languages)

best_genre = top_3_genres.iloc[0]
best_game = top_5_games.iloc[0]
best_developer = top_5_developers.iloc[0]
phase_1_languages = ", ".join(top_5_languages["language"].tolist())

display(Markdown(f"""
# Executive Recommendation

## 1. Genre Priority

Genre nên ưu tiên nghiên cứu sâu là **{best_genre["game_genre"]}**.

Lý do:

- Free Genre Attractiveness Score cao nhất: **{best_genre["free_genre_attractiveness_score"]:.1f}**
- Median rating velocity: **{best_genre["median_rating_velocity"]:,.0f}**
- Average weighted rating: **{best_genre["avg_weighted_rating"]:.2f}**
- Freshness rate: **{best_genre["genre_freshness_rate"]:.1%}**
- Opportunity type: **{best_genre["opportunity_type"]}**

Điều này cho thấy genre này có tín hiệu tốt về traction, rating quality và mức độ active gần đây.

## 2. Benchmark Strategy

Game nên benchmark đầu tiên là **{best_game["name"]}** của **{best_game["developer"]}**.

Developer nên benchmark là **{best_developer["developer"]}**.

Không nên benchmark chỉ dựa trên average rating. Nên benchmark những game/developer có:

- Rating count cao.
- Weighted rating tốt.
- Rating velocity tốt.
- Có dấu hiệu localization phù hợp.

## 3. Localization Strategy

Không nên localization quá rộng ngay từ đầu.

Phase 1 nên ưu tiên các ngôn ngữ xuất hiện nhiều trong nhóm high-traction Free games:

**{phase_1_languages}**

Sau khi game có traction thật, mới mở rộng thêm ngôn ngữ ở Phase 2.

## 4. Launch Strategy

Nếu studio muốn launch theo hướng **viral/casual ngắn hạn**:

- Ưu tiên gameplay dễ hiểu.
- Giảm friction tải game.
- Tập trung rating velocity và early traction.
- Test ads/IAP sớm.

Nếu studio muốn launch theo hướng **live-service dài hạn**:

- Cần bổ sung dữ liệu retention, engagement, IAP, event performance sau launch.
- Cần đầu tư community, update định kỳ, event và localization sâu hơn.
- Không thể quyết định live-service chỉ bằng dataset hiện tại.

# 2–3 Priority Actions for Stakeholder

1. **Ưu tiên validate genre {best_genre["game_genre"]}** bằng competitor review, gameplay benchmark và prototype test.
2. **Benchmark top games/developers** trong bảng trên để học về positioning, localization và traction strategy.
3. **Triển khai localization theo phase**, bắt đầu với nhóm ngôn ngữ có tín hiệu cao, sau đó mở rộng khi có traction thật.

# Important Limitation

Các recommendation trên dựa trên proxy metrics từ App Store rating data.

Dataset chưa có download, revenue, IAP, ads revenue, retention, DAU/MAU và marketing spend. Vì vậy, đây là định hướng phân tích để hỗ trợ quyết định ban đầu, không phải kết luận tuyệt đối về khả năng thành công thương mại.
"""))


# 20. Report Flow Suggestion

Business request không yêu cầu dashboard, nên không tạo dashboard.

Nếu cần trình bày thành report hoặc slide, nên dùng flow storytelling như sau:

## Page 1 — Executive Summary

Thông điệp chính:

> Nếu launch game Free, stakeholder nên ưu tiên genre có traction velocity tốt, weighted rating ổn và còn active trong giai đoạn gần nhất.

Nội dung:

- Top genre recommendation.
- Top benchmark game.
- Top benchmark developer.
- Localization phase 1 recommendation.
- 2–3 priority actions.

## Page 2 — Business Question & Dataset Limitation

Thông điệp chính:

> Dataset chỉ có App Store rating data, nên các chỉ số đều là proxy.

Nội dung:

- Business question.
- Scope: Free games only.
- Recent period: 3 năm gần nhất.
- Limitation: không có download/revenue/retention/IAP.

## Page 3 — Genre Opportunity

Thông điệp chính:

> Không chọn genre chỉ vì nhiều game hoặc rating cao, mà cần kết hợp traction, quality, freshness và competition.

Nội dung:

- Genre Attractiveness Ranking.
- Genre Opportunity Matrix.
- Recommendation genre.

## Page 4 — Benchmark Game & Developer

Thông điệp chính:

> Benchmark nên dựa trên rating count, weighted rating và rating velocity, không dựa trên average rating đơn thuần.

Nội dung:

- Top Free Benchmark Games.
- Top Developer Benchmark.
- Case Study.

## Page 5 — Localization Strategy

Thông điệp chính:

> Localization nên triển khai theo giai đoạn, không mở rộng quá nhiều ngay từ đầu.

Nội dung:

- Localization group performance.
- Top languages among high-traction games.
- Phase 1 / Phase 2 recommendation.

## Page 6 — Final Recommendation

Thông điệp chính:

> Launch strategy nên bắt đầu bằng validate genre + benchmark + phased localization; sau launch mới bổ sung retention/IAP để quyết định viral hay live-service dài hạn.

Nội dung:

- 2–3 priority actions.
- Required additional data after launch.
